# Orthogonal constraint steering — Kaggle runnerEverything runs inside the notebook kernel, so the model is downloaded once andloaded once. Re-running the generate cell reuses the model already in memory.Every story is written to `state.json` the moment it is produced, so an interruptat any point costs you nothing: re-run the cell and it continues.Accelerator must be **GPU T4 x2**.

In [ ]:
MODEL   = "AceGPT"                       # Fanar | ALLaM | AceGPT | Jais | Phi-4-miniSUITE   = "method"                       # method = new method only (quick test)                                         # core   = + Baseline and L-Res references                                         # ortho | beta | loo | allSTORIES = 50                             # stories per conditionREPO    = "/kaggle/working/NoiseEGRA"BRANCH  = "NoiseSteering"# One folder per model. The activation scale differs ~30x between models,# and the scoring script scores every CSV in whichever folder you point it at.OUT     = f"/kaggle/working/orthosteer/{MODEL}"

## 1. Install and clone

In [ ]:
!pip install -q -U "transformers>=4.56" accelerate hf_transfer!rm -rf {REPO} && git clone -q --branch {BRANCH} --single-branch \    https://github.com/haziq-exe/NoiseEGRA.git {REPO}import os, sysos.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"     # multi-threaded downloaderos.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"      # default is 10s; too short for big shardssys.path[:0] = [REPO, f"{REPO}/scripts"]!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader!df -h /kaggle/working | tail -1

## 2. Hugging Face loginFanar, ALLaM and Jais are gated. Add your token under **Add-ons → Secrets** as `HF_TOKEN`.

In [ ]:
from kaggle_secrets import UserSecretsClientfrom huggingface_hub import loginlogin(UserSecretsClient().get_secret("HF_TOKEN"))

## 3. Download the weightsSeparate from loading, so a failed download is a cheap retry. It resumes: filesalready fetched are skipped, so **just re-run this cell if it stalls**.If it repeatedly stops at the same percentage, the next cell shows which singlefile is the problem.

In [ ]:
from huggingface_hub import snapshot_downloadfrom noiseegra.defaults import MODEL_HF_IDSpath = snapshot_download(    MODEL_HF_IDS[MODEL],    local_dir=f"{WEIGHTS}/{MODEL}",    max_workers=8,    allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt", "tokenizer*"],)print("weights at", path)

In [ ]:
# Only if the download keeps stalling: shows which files are still partial.!du -sh {WEIGHTS}/{MODEL} 2>/dev/null!find {WEIGHTS}/{MODEL} -name "*.incomplete" -exec ls -lh {{}} \; 2>/dev/null | head!ls -lh {WEIGHTS}/{MODEL}/*.safetensors 2>/dev/null

## 4. Load the modelRun once. The model then stays in memory for as many generate calls as you like.

In [ ]:
from kaggle_orthosteer import load_model, run, make_argsmodel = load_model(MODEL, model_id=f"{WEIGHTS}/{MODEL}")

## 5. GenerateSafe to interrupt. Re-run to resume — the model is not reloaded.

In [ ]:
run(model, make_args(model=MODEL, suite=SUITE, num_stories=STORIES, out=OUT))

## 6. Score

In [ ]:
!cd {REPO} && python -u scripts/score_orthosteer.py --input-dir {OUT}

Results are in `/kaggle/working/orthosteer/`:| file | what it is ||---|---|| `state.json` | the checkpoint, and the archive of every story generated || `<condition>.csv` | one story per row || `EXACT_SCORES/Ortho_Constraint_Table.md` | comparison table across conditions |To carry work into a new session: **Save Version**, then next time add thatversion's output via **Add-ons → Add data → Your Work** and copy `state.json`(and the `weights/` folder) back into `/kaggle/working`.